<a href="https://colab.research.google.com/github/nelsbuhrley/CSE-450-Machine-learning/blob/main/notebooks/module01-assessment-polars.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn as skl

# Introduction
This assignment will test how well you're able to perform various data science-related tasks.

Each Problem Group below will center around a particular dataset that you have worked with before.

To ensure you receive full credit for a question, make sure you demonstrate the appropriate polars, lets-plot, or other commands as requested in the provided code blocks.

You may find that some questions require multiple steps to fully answer. Others require some mental arithmetic in addition to polars commands. Use your best judgment.

## Submission
Each problem group asks a series of questions. This assignment consists of two submissions:

1. After completing the questions below, open the Module 01 Assessment Quiz in Canvas and enter your answers to these questions there.

2. After completing and submitting the quiz, save this Colab notebook as a GitHub Gist (You'll need to create a GitHub account for this), by selecting `Save a copy as a GitHub Gist` from the `File` menu above.

    In Canvas, open the Module 01 Assessment GitHub Gist assignment and paste the GitHub Gist URL for this notebook. Then submit that assignment.

## Problem Group 1

For the questions in this group, you'll work with the Netflix Movies Dataset found at this url: [https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/netflix_titles.csv](https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/netflix_titles.csv)


### Question 1
Load the dataset into a Polars data frame and determine what data type is used to store the `release_year` feature.

In [30]:
data_netflix = pl.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/netflix_titles.csv')
data_netflix.select('release_year').head(0)
data_netflix.head(4)

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
i64,str,str,str,str,str,str,i64,str,str,str,str
81145628,"""Movie""","""Norm of the North: King Sized …","""Richard Finn, Tim Maltby""","""Alan Marriott, Andrew Toth, Br…","""United States, India, South Ko…","""September 9, 2019""",2019,"""TV-PG""","""90 min""","""Children & Family Movies, Come…","""Before planning an awesome wed…"
80117401,"""Movie""","""Jandino: Whatever it Takes""",null,"""Jandino Asporaat""","""United Kingdom""","""September 9, 2016""",2016,"""TV-MA""","""94 min""","""Stand-Up Comedy""","""Jandino Asporaat riffs on the …"
70234439,"""TV Show""","""Transformers Prime""",null,"""Peter Cullen, Sumalee Montano,…","""United States""","""September 8, 2018""",2013,"""TV-Y7-FV""","""1 Season""","""Kids' TV""","""With the help of three human a…"
80058654,"""TV Show""","""Transformers: Robots in Disgui…",null,"""Will Friedle, Darren Criss, Co…","""United States""","""September 8, 2018""",2016,"""TV-Y7""","""1 Season""","""Kids' TV""","""When a prison ship crash unlea…"


The datatype of release_year is a 64 bit intiger

### Question 2
Filter your dataset so it contains only `TV Shows`. How many of those TV Shows were rated `TV-Y7`?

In [31]:
data_netflix = data_netflix.filter(pl.col('type')=='TV Show')
data_netflix.filter(pl.col('rating')=='TV-Y7').shape[0]

100

### Question 3
Further filter your dataset so it only contains TV Shows released between the years 2000 and 2009 inclusive. How many of *those* shows were rated `TV-Y7`?

In [34]:
data_netflix = data_netflix.filter((pl.col('release_year').is_between(2000,2009)))
data_netflix.filter(pl.col('rating')=='TV-Y7').shape[0]

4

## Problem Group 2

For the questions in this group, you'll work with the Cereal Dataset found at this url: [https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/cereal.csv](https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/cereal.csv)


### Question 4
After importing the dataset into a polars data frame, determine the median amount of `protein` in cereal brands manufactured by Kelloggs. (`mfr` code "K")

In [37]:
data_cerial = pl.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/cereal.csv')
data_cerial.head(4)

name,mfr,type,calories,protein,fat,sodium,fiber,carbo,sugars,potass,vitamins,shelf,weight,cups,rating
str,str,str,i64,i64,i64,i64,f64,f64,i64,i64,i64,i64,f64,f64,f64
"""100% Bran""","""N""","""C""",70,4,1,130,10.0,5.0,6,280,25,3,1.0,0.33,68.402973
"""100% Natural Bran""","""Q""","""C""",120,3,5,15,2.0,8.0,8,135,0,3,1.0,1.0,33.983679
"""All-Bran""","""K""","""C""",70,4,1,260,9.0,7.0,5,320,25,3,1.0,0.33,59.425505
"""All-Bran with Extra Fiber""","""K""","""C""",50,4,0,140,14.0,8.0,0,330,25,3,1.0,0.5,93.704912


In [42]:
data_cerial.filter(pl.col('mfr')=='K').select(pl.col('protein')).median()

protein
f64
3.0


### Question 5
In order to comply with new government regulations, all cereals must now come with a "Healthiness" rating. This rating is calculated based on this formula:

    healthiness = (protein + fiber) / sugar

Create a new `healthiness` column populated with values based on the above formula.

Then, determine the median healthiness value for only General Mills cereals (`mfr` = "G"), rounded to two decimal places.

In [47]:
data_cerial = data_cerial.with_columns(
    ((pl.col('protein') + pl.col('fiber')) / pl.col('sugars')).alias('healthiness')
)

print(data_cerial.filter(pl.col('mfr')=='G').select(pl.col('healthiness')).median())

shape: (1, 1)
┌─────────────┐
│ healthiness │
│ ---         │
│ f64         │
╞═════════════╡
│ 0.475       │
└─────────────┘


## Problem Group 3

For the questions in this group, you'll work with the Titanic Dataset found at this url: [https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/titanic.csv](https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/titanic.csv)

### Question 6

After loading the dataset into a polars DataFrame, create a new column called `NameGroup` that contains the first letter of the passenger's surname in lower case.

Note that in the dataset, passenger's names are provided in the `Name` column and are listed as:

    Surname, Given names

For example, if a passenger's `Name` is `Braund, Mr. Owen Harris`, the `NameGroup` column should contain the value `b`.

Then count how many passengers have a `NameGroup` value of `k`.

In [52]:
data_titanic = pl.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/titanic.csv')
data_titanic.head(4)

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,str,i64,str,str,f64,i64,i64,str,f64,str,str
1,"""No""",3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,"""Yes""",1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,"""Yes""",3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,"""Yes""",1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""


In [53]:
data_titanic = data_titanic.with_columns(
    pl.col('Name').str.to_lowercase()
)
data_titanic = data_titanic.with_columns(
    pl.col('Name').str.slice(0,1).alias('NameGroup')
)

data_titanic.head()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,NameGroup
i64,str,i64,str,str,f64,i64,i64,str,f64,str,str,str
1,"""No""",3,"""braund, mr. owen harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S""","""b"""
2,"""Yes""",1,"""cumings, mrs. john bradley (fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""","""c"""
3,"""Yes""",3,"""heikkinen, miss. laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""","""h"""
4,"""Yes""",1,"""futrelle, mrs. jacques heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S""","""f"""
5,"""No""",3,"""allen, mr. william henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S""","""a"""


In [54]:
print(data_titanic.filter(pl.col('NameGroup')=='k').shape[0])

28


There are